# 02 · CRISPR benchmark: Table 1a (K562, HCT116, GM12878, WTC11) and Table 1c (by distance)

Held-out CRISPR element–gene pairs from the ENCODE CRISPR benchmark: K562 pairs from five studies not used in training (1,895 pairs, 115 positives), HCT116 (373, 34), GM12878 (68, 16) and WTC11 DC-TAP (1,908, 15; no histone marks, so CisLink-V2G only). CisLink scores are computed here from the shipped feature tables; ENCODE-rE2G, ABC and distance come with the pairs. 95% CIs: gene-clustered bootstrap (1,000 replicates).

The hFOB CRISPRi and STING-seq screens of BMD variants are not redistributed here; see `data/README.md`.

In [1]:
import numpy as np
import pandas as pd

from cislink import datasets
from cislink import evaluate as ev

M = ["CisLink-E2G", "CisLink-V2G", "ENCODE-rE2G Extended", "ENCODE-rE2G", "ABC", "Distance"]
P = {c: datasets.crispr_pairs(c) for c in datasets.CELLS}
rows = []
for c, p in P.items():
    r = {"set": c, "pairs": len(p), "positives": int(p.label.sum())}
    for m in M:
        r[m] = np.nan if p[m].isna().all() else ev.auprc(p.label, p[m])
    rows.append(r)
pd.DataFrame(rows).round(3)

,set,pairs,positives,CisLink-E2G,CisLink-V2G,ENCODE-rE2G Extended,ENCODE-rE2G,ABC,Distance
0,K562,1895,115,0.700,0.603,0.678,0.632,0.519,0.366
1,HCT116,373,34,0.750,0.513,NaN,0.593,NaN,0.324
2,GM12878,68,16,0.913,0.880,0.669,0.653,0.731,0.853
3,WTC11,1908,15,NaN,0.429,NaN,0.393,0.336,0.458


In [2]:
ci = []
for c, p in P.items():
    for m in ("CisLink-E2G", "CisLink-V2G"):
        if p[m].isna().all():   # WTC11: CisLink-E2G not scored (no histone marks)
            continue
        a, lo, hi = ev.auprc_ci(p.label, p[m], p.cluster)
        ci.append({"set": c, "model": m, "AUPRC": a, "lo": lo, "hi": hi})
pd.DataFrame(ci).round(3)

,set,model,AUPRC,lo,hi
0,K562,CisLink-E2G,0.700,0.608,0.782
1,K562,CisLink-V2G,0.603,0.490,0.707
2,HCT116,CisLink-E2G,0.750,0.649,0.887
3,HCT116,CisLink-V2G,0.513,0.291,0.754
4,GM12878,CisLink-E2G,0.913,0.772,1.000
5,GM12878,CisLink-V2G,0.880,0.722,1.000
6,WTC11,CisLink-V2G,0.429,0.146,0.704


## Paired differences on the same pairs

In [3]:
d = []
for c, p in P.items():
    for a, b in (("CisLink-E2G", "ENCODE-rE2G Extended"), ("CisLink-E2G", "ENCODE-rE2G"), ("CisLink-E2G", "CisLink-V2G"), ("CisLink-E2G", "Distance")):
        if p[a].isna().all() or p[b].isna().all():
            continue
        diff, lo, hi = ev.paired_difference(p.label, p[a], p[b].fillna(0), p.cluster)
        d.append({"set": c, "comparison": f"{a} - {b}", "difference": diff, "lo": lo, "hi": hi})
pd.DataFrame(d).round(3)

,set,comparison,difference,lo,hi
0,K562,CisLink-E2G - ENCODE-rE2G Extended,0.022,-0.041,0.090
1,K562,CisLink-E2G - ENCODE-rE2G,0.068,0.005,0.139
2,K562,CisLink-E2G - CisLink-V2G,0.097,0.044,0.152
3,K562,CisLink-E2G - Distance,0.333,0.234,0.416
4,HCT116,CisLink-E2G - ENCODE-rE2G,0.157,0.049,0.296
5,HCT116,CisLink-E2G - CisLink-V2G,0.237,0.057,0.407
6,HCT116,CisLink-E2G - Distance,0.426,0.283,0.588
7,GM12878,CisLink-E2G - ENCODE-rE2G Extended,0.244,0.013,0.423
8,GM12878,CisLink-E2G - ENCODE-rE2G,0.260,0.000,0.414
9,GM12878,CisLink-E2G - CisLink-V2G,0.033,-0.064,0.196


## Table 1c: held-out K562 pairs by element–TSS distance

CisLink-E2G's margin over distance is largest 10–200 kb from the TSS; its margin over CisLink-V2G is significant 10–50 kb from the TSS.

In [4]:
k = P["K562"].copy(); k["bin"] = ev.distance_bins(k.distance_to_TSS)
t = []
for lo_, hi_, b in ev.DISTANCE_BINS:
    q = k[k.bin == b]
    r = {"distance": b, "pairs": len(q), "positives": int(q.label.sum())}
    for m in ("CisLink-E2G", "CisLink-V2G", "ENCODE-rE2G Extended", "Distance"):
        r[m] = ev.auprc(q.label, q[m])
    for a, b2 in (("CisLink-E2G", "Distance"), ("CisLink-E2G", "CisLink-V2G")):
        diff, lo2, hi2 = ev.paired_difference(q.label, q[a], q[b2], q.cluster)
        r[f"{a.split('-')[1]} - {b2.replace('CisLink-', '')}"] = f"{diff:+.3f} [{lo2:+.3f}, {hi2:+.3f}]"
    t.append(r)
pd.DataFrame(t).round(3)

,distance,pairs,positives,CisLink-E2G,CisLink-V2G,ENCODE-rE2G Extended,Distance,E2G - Distance,E2G - V2G
0,<10 kb,31,17,0.918,0.855,0.913,0.753,"+0.165 [-0.024, +0.301]","+0.063 [-0.030, +0.198]"
1,10-50 kb,135,40,0.836,0.706,0.831,0.414,"+0.423 [+0.260, +0.537]","+0.131 [+0.032, +0.224]"
2,50-200 kb,396,36,0.689,0.603,0.659,0.158,"+0.531 [+0.352, +0.670]","+0.086 [-0.007, +0.159]"
3,>200 kb,1333,22,0.219,0.135,0.262,0.035,"+0.184 [+0.032, +0.387]","+0.084 [-0.005, +0.176]"


## Pooled over cell types not used for training

The manuscript pools four sets (HCT116, GM12878 and the two hFOB screens) as the positive-weighted mean of per-set AUPRC differences. With the two redistributable sets:

In [5]:
S = {c: P[c].rename(columns={"label": "y"}) for c in ("HCT116", "GM12878")}
for a, b in (("CisLink-E2G", "CisLink-V2G"), ("CisLink-E2G", "ENCODE-rE2G")):
    r = ev.pooled_difference(S, a, b)
    print(f"{a} - {b}: {r['difference']:+.3f} [{r['lo']:+.3f}, {r['hi']:+.3f}]  per set {', '.join(f'{k} {v:+.3f}' for k, v in r['per_set'].items())}")

CisLink-E2G - CisLink-V2G: +0.172 [+0.044, +0.294]  per set HCT116 +0.237, GM12878 +0.033


CisLink-E2G - ENCODE-rE2G: +0.190 [+0.068, +0.293]  per set HCT116 +0.157, GM12878 +0.260
